In [14]:
from PIL import Image
from PIL.ExifTags import TAGS

photo = "IMG_4615.JPEG"

img = Image.open(photo)

for tag_id, value in img.getexif().items():
    print(TAGS.get(tag_id, tag_id), value)

GPSInfo 2560
ResolutionUnit 2
ExifOffset 214
Make Apple
Model iPhone 13
Software 26.6
DateTime 2026:09:18 06:52:09
YCbCrPositioning 1
XResolution 72.0
YResolution 72.0
HostComputer iPhone 13


In [15]:
from PIL import Image
from PIL.ExifTags import GPSTAGS

photo = "IMG_4615.JPEG"

img = Image.open(photo)
exif = img.getexif()

gps_info = exif.get(34853)   # GPSInfo-tag

print(gps_info)

2560


In [16]:
import exifread

with open("IMG_4615.JPEG", "rb") as f:
    tags = exifread.process_file(f)

for tag in tags:
    if "GPS" in tag:
        print(tag, tags[tag])

GPS GPSLatitudeRef N
GPS GPSLatitude [21, 34, 241/50]
GPS GPSLongitudeRef W
GPS GPSLongitude [88, 14, 146/25]
GPS GPSAltitudeRef 0
GPS GPSAltitude 29004/6487
GPS GPSTimeStamp [12, 52, 9]
GPS GPSSpeedRef K
GPS GPSSpeed 0
GPS GPSImgDirectionRef T
GPS GPSImgDirection 181517/1224
GPS GPSDestBearingRef T
GPS GPSDestBearing 181517/1224
GPS GPSDate 2026:09:18
GPS Tag 0x001F 108502/30689
Image GPSInfo 2560


In [17]:
from PIL import Image
from PIL.ExifTags import TAGS
from pathlib import Path

photo_folder = Path("Photos")

for photo in photo_folder.glob("*.JPEG"):

    img = Image.open(photo)
    exif = img.getexif()

    gps_found = False

    for tag_id, value in exif.items():

        tag = TAGS.get(tag_id, tag_id)

        if tag == "GPSInfo":
            gps_found = True
            print(photo.name, "contains GPS data")

    if not gps_found:
        print(photo.name, "contains NO GPS data")



5FD3464C-1A53-454B-9EC5-83E2F8EA42AC.JPEG contains NO GPS data
IMG_4615.JPEG contains GPS data
IMG_4616.JPEG contains GPS data
IMG_4617.JPEG contains GPS data
IMG_4619.JPEG contains GPS data
IMG_4623.JPEG contains GPS data
IMG_4624.JPEG contains GPS data
IMG_4649.JPEG contains GPS data
IMG_4651.JPEG contains GPS data
IMG_4652.JPEG contains GPS data
IMG_4653.JPEG contains GPS data


In [18]:
import exifread

def convert_to_decimal(coord, ref):

    d = float(coord.values[0])
    m = float(coord.values[1])
    s = float(coord.values[2])

    decimal = d + m/60 + s/3600

    if ref in ["S", "W"]:
        decimal = -decimal

    return decimal


import exifread

with open("IMG_4615.JPEG", "rb") as f:

    tags = exifread.process_file(f)

    lat = convert_to_decimal(
        tags["GPS GPSLatitude"],
        str(tags["GPS GPSLatitudeRef"])
    )

    lon = convert_to_decimal(
        tags["GPS GPSLongitude"],
        str(tags["GPS GPSLongitudeRef"])
    )

print(lat, lon)

21.568005555555555 -88.23495555555556


In [19]:
from pathlib import Path
import exifread
import pandas as pd

records = []

for photo in Path("Photos").glob("*.JPEG"):

    with open(photo, "rb") as f:

        tags = exifread.process_file(f)

        try:

            lat = convert_to_decimal(
                tags["GPS GPSLatitude"],
                str(tags["GPS GPSLatitudeRef"])
            )

            lon = convert_to_decimal(
                tags["GPS GPSLongitude"],
                str(tags["GPS GPSLongitudeRef"])
            )

            records.append({
                "photo": photo.name,
                "latitude": lat,
                "longitude": lon
            })

        except:
            print(f"No GPS found in {photo.name}")

df = pd.DataFrame(records)

df.to_excel(
    "building_inventory.xlsx",
    index=False
)

print(df.head())

No GPS found in 5FD3464C-1A53-454B-9EC5-83E2F8EA42AC.JPEG
           photo   latitude  longitude
0  IMG_4615.JPEG  21.568006 -88.234956
1  IMG_4616.JPEG  21.568008 -88.235092
2  IMG_4617.JPEG  21.567956 -88.234733
3  IMG_4619.JPEG  21.567578 -88.234619
4  IMG_4623.JPEG  21.566764 -88.234803


In [20]:
import folium
import pandas as pd

# Excelbestand uit de vorige stap
df = pd.read_excel("building_inventory.xlsx")

# Centreer de kaart op alle foto's
center_lat = df["latitude"].mean()
center_lon = df["longitude"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=17,
    tiles=None
)

folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google',
    name='Google Satellite',
    overlay=False,
    control=True
).add_to(m)

# Voeg markers toe
for _, row in df.iterrows():

    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=row["photo"],
        tooltip=row["photo"]
    ).add_to(m)

# Opslaan
m.save("SanFelipe_Photos.html")

print("Kaart opgeslagen!")


Kaart opgeslagen!


In [21]:
import folium
import pandas as pd

df = pd.read_excel("building_inventory.xlsx")

m = folium.Map(
    location=[df.latitude.mean(), df.longitude.mean()],
    zoom_start=17,
    tiles=None
)

folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google",
    name="Google Satellite"
).add_to(m)

for _, row in df.iterrows():
    photo_path = f"Photos/{row['photo']}"

    html = f"""
    <h4>{row['photo']}</h4>
    <img src="{photo_path}" width="300">
    """

    popup = folium.Popup(html, max_width=350)

    folium.Marker(
        [row["latitude"], row["longitude"]],
        popup=popup,
        tooltip=row["photo"]
    ).add_to(m)

m.save("SanFelipe_Photos.html")
